# 🎤 MedPilot — PhoWhisper STT Server

**Chạy Whisper (faster-whisper) thành REST API qua Ngrok**

### Yêu cầu
- Runtime: **GPU (T4 hoặc tốt hơn)**  
- Chạy lần lượt từng cell theo thứ tự

### Endpoint sau khi chạy xong
```
POST {ngrok_url}/v1/audio/transcriptions
```

In [ ]:
# ── Cell 1: Cài đặt thư viện ──────────────────────────────
!pip install faster-whisper pyngrok fastapi uvicorn python-multipart -q
print('✅ Cài đặt xong!')

In [ ]:
# ── Cell 2: Load Whisper model ────────────────────────────
from faster_whisper import WhisperModel

print('🚀 Đang load Whisper model (medium)...')
print('   (Lần đầu cần ~3-5 phút để download)')

whisper_model = WhisperModel(
    model_size_or_path='medium',
    device='cuda',
    compute_type='float16',          # float16 cho GPU T4
    download_root='/root/.cache/whisper'
)

print('✅ Whisper model loaded!')

In [ ]:
# ── Cell 3: Tạo FastAPI app ───────────────────────────────
import os
import tempfile
import threading
import uvicorn
from fastapi import FastAPI, HTTPException, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title='MedPilot Whisper STT', version='2.0')

# CORS — cho phép mọi nguồn
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['*'],
    allow_headers=['*'],
)


@app.get('/')
def root():
    return {
        'status': 'running',
        'service': 'MedPilot Whisper STT',
        'endpoints': {
            'transcribe': 'POST /v1/audio/transcriptions',
            'health':     'GET  /health'
        }
    }


@app.get('/health')
def health():
    return {'status': 'healthy', 'model': 'faster-whisper-medium'}


# ─────────────────────────────────────────────────────────
@app.post('/v1/audio/transcriptions')
async def transcribe_audio(
    file: UploadFile = File(...)
):
    """
    Nhận file audio → trả về transcript tiếng Việt.

    Form-data fields:
        file     : audio file (.webm / .mp3 / .wav / .m4a)
        model    : (ignored — always uses loaded model)
        language : (ignored — always vi)
        task     : (ignored — always transcribe)

    Returns:
        { text, language, duration, segments }
    """
    # Chấp nhận mọi định dạng phổ biến
    ALLOWED_EXTS = {'.webm', '.mp3', '.wav', '.m4a', '.ogg', '.flac', '.mp4'}
    fname = (file.filename or 'audio.webm').lower()
    ext   = os.path.splitext(fname)[1] or '.webm'
    if ext not in ALLOWED_EXTS:
        ext = '.webm'   # fallback an toàn

    tmp_path = None
    try:
        audio_bytes = await file.read()
        if not audio_bytes:
            raise HTTPException(status_code=400, detail='File audio rỗng')

        print(f'📥 Nhận file: {file.filename} ({len(audio_bytes):,} bytes)')

        # Ghi tạm ra disk (faster-whisper cần đường dẫn file)
        with tempfile.NamedTemporaryFile(delete=False, suffix=ext) as tmp:
            tmp.write(audio_bytes)
            tmp_path = tmp.name

        # Chạy STT
        segments_iter, info = whisper_model.transcribe(
            tmp_path,
            language='vi',
            task='transcribe',
            beam_size=5,
            vad_filter=True,          # lọc silence
            vad_parameters={'min_silence_duration_ms': 500}
        )

        segs = list(segments_iter)    # materialise generator
        text = ' '.join(s.text.strip() for s in segs if s.text.strip())

        segments_out = [
            {'id': s.id, 'start': round(s.start, 2),
             'end': round(s.end, 2), 'text': s.text.strip()}
            for s in segs
        ]

        print(f'✅ Transcript ({len(text)} chars): {text[:120]}...')

        return {
            'text':     text,
            'language': info.language,
            'duration': round(info.duration, 2),
            'segments': segments_out
        }

    except HTTPException:
        raise
    except Exception as e:
        print(f'❌ Lỗi: {e}')
        raise HTTPException(status_code=500, detail=f'Whisper error: {str(e)}')
    finally:
        if tmp_path and os.path.exists(tmp_path):
            os.unlink(tmp_path)


print('✅ FastAPI app đã định nghĩa!')

In [ ]:
# ── Cell 4: Khởi động server + tạo Ngrok tunnel ──────────
from pyngrok import ngrok
import time

# ⚠️ Điền Ngrok auth token của bạn vào đây
NGROK_TOKEN = '3BM0sCxW0eDMEgAhshblG0Q6cP8_3rdRpdmFj8vkh8oojfZvA'
ngrok.set_auth_token(NGROK_TOKEN)

# Đóng tunnel cũ nếu có
for t in ngrok.get_tunnels():
    try:
        ngrok.disconnect(t.public_url)
    except:
        pass

# Khởi server trong background thread
def run_server():
    config = uvicorn.Config(app, host='0.0.0.0', port=8001, log_level='warning')
    server = uvicorn.Server(config)
    server.run()

t = threading.Thread(target=run_server, daemon=True)
t.start()
time.sleep(2)   # chờ server lên

# Tạo tunnel
tunnel = ngrok.connect(8001, bind_tls=True)
public_url = tunnel.public_url

print()
print('=' * 65)
print('🎉  WHISPER STT SERVER SẴN SÀNG!')
print('=' * 65)
print()
print(f'🌐 Ngrok URL : {public_url}')
print()
print('📋 COPY VÀO FILE .env:')
print('-' * 65)
print(f'WHISPER_API_URL={public_url}/v1/audio/transcriptions')
print('-' * 65)
print()
print('🧪 Test nhanh (chạy trong terminal trên máy local):')
print(f'   curl -X POST "{public_url}/v1/audio/transcriptions" \\')
print( '        -H "ngrok-skip-browser-warning: true" \\')
print( '        -F "file=@audio.wav"')
print()
print('⚠️  Giữ cell này chạy — đóng tab = mất kết nối!')